In [1]:
import sys
import os

# Go to project root
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Add dtgraph folder to path
sys.path.append(os.path.join(project_root, "../dtgraph"))

In [2]:
from dtgraph import Neo4jGraph, Rule, Transformation

hostname = "localhost"
password = "internship"
uri = f"bolt://{hostname}:7687"

graph = Neo4jGraph(uri, database="neo4j", username="neo4j", password=password)

In [3]:
from type_checking.environment import Environment

env = Environment("../../dtgraph/type_checking/ENVs/env_panama_experimentation.json")

In [4]:
from pg_schema.loader import SchemaLoader

source_schema_path = "../../dtgraph/pg_schema/schemas/schema_panama_source.json"
target_schema_path = "../../dtgraph/pg_schema/schemas/schema_panama_target.json"

source_schema = SchemaLoader(
    source_schema_path,
    env=env,
    section="source"
)

target_schema = SchemaLoader(
    target_schema_path,
    env=env,
    section="target"
)

### Rules

In [5]:
Rule1_A = Rule(
    """
MATCH (e:Entity)
WHERE NOT e:StruckOff
GENERATE
(ae = (e):ActiveEntity {
    name = e.name,
    country_codes = e.country_codes,
    incorporation_date = e.incorporation_date,
    sourceID = e.sourceID
})
""",
    env=env,
    type_strict=False,
)

Rule1_B = Rule(
    """
MATCH (e:Entity:StruckOff)
GENERATE
(ie = (e):InactiveEntity {
    name = e.name,
    country_codes = e.country_codes,
    incorporation_date = e.incorporation_date,
    sourceID = e.sourceID,
    inactivation_date = e.struck_off_date,
})
""",
    env=env,
    type_strict=False,
)

Rule2_A = Rule(
    """
MATCH (e:Entity)-[r:registered_address]->(a:Address)
WHERE NOT e:StruckOff AND a.address IS NOT NULL
GENERATE
(ae = (e):ActiveEntity {
    name = e.name,
    country_codes = e.country_codes,
    incorporation_date = e.incorporation_date,
    sourceID = e.sourceID
})
-[():REGISTERED_AT {
    link = r.link
}]->
(ra = (a.address):RegisteredAddress {
    address = a.address,
    country_codes = a.country_codes,
    name = a.name
})
""",
    env=env,
    type_strict=False,
)

Rule2_B = Rule(
    """
MATCH (e:Entity:StruckOff)-[r:registered_address]->(a:Address)
WHERE a.address IS NOT NULL
GENERATE
(ie = (e):InactiveEntity {
    name = e.name,
    country_codes = e.country_codes,
    incorporation_date = e.incorporation_date,
    sourceID = e.sourceID,
    inactivation_date = e.struck_off_date,
})
-[():REGISTERED_AT {
    link = r.link
}]->
(ra = (a.address):RegisteredAddress {
    address = a.address,
    country_codes = a.country_codes,
    name = a.name
})
""",
    env=env,
    type_strict=False,
)

Rule3_A = Rule(
    """
MATCH (o:Officer)-[r:officer_of]->(e:Entity)
WHERE NOT e:StruckOff AND o.name IS NOT NULL
GENERATE
(op = (o.name):OfficerProfile {
    name = o.name,
    sourceID = o.sourceID
})
-[():OFFICER_OF {
    link = r.link,
    start_date = r.start_date,
    sourceID = r.sourceID
}]->
(ae = (e):ActiveEntity {
    name = e.name,
    country_codes = e.country_codes,
    incorporation_date = e.incorporation_date,
    sourceID = e.sourceID
})
""",
    env=env,
    type_strict=False,
)

Rule3_B = Rule(
    """
MATCH (o:Officer)-[r:officer_of]->(e:Entity:StruckOff)
WHERE o.name IS NOT NULL
GENERATE
(op = (o.name):OfficerProfile {
    name = o.name,
    sourceID = o.sourceID
})
-[():OFFICER_OF {
    link = r.link,
    start_date = r.start_date,
    sourceID = r.sourceID
}]->
(ie = (e):InactiveEntity {
    name = e.name,
    country_codes = e.country_codes,
    incorporation_date = e.incorporation_date,
    sourceID = e.sourceID,
    inactivation_date = e.struck_off_date,
})
""",
    env=env,
    type_strict=False,
)

Rule4_A = Rule(
    """
MATCH (i:Intermediary)-[r:intermediary_of]->(e:Entity)
WHERE NOT e:StruckOff AND i.address IS NOT NULL
GENERATE
(ip = (i.address):IntermediaryProfile {
    name = i.name,
    address = i.address,
    status = i.status,
    country_codes = i.country_codes
})
-[():INTERMEDIARY_OF {
    link = r.link,
}]->
(ae = (e):ActiveEntity {
    name = e.name,
    country_codes = e.country_codes,
    incorporation_date = e.incorporation_date,
    sourceID = e.sourceID
})
""",
    env=env,
    type_strict=False,
)


Rule4_B = Rule(
    """
MATCH (i:Intermediary)-[r:intermediary_of]->(e:Entity:StruckOff)
WHERE i.address IS NOT NULL
GENERATE
(ip = (i.address):IntermediaryProfile {
    name = i.name,
    address = i.address,
    status = i.status,
    country_codes = i.country_codes
})
-[():INTERMEDIARY_OF {
    link = r.link
}]->
(ie = (e):InactiveEntity {
    name = e.name,
    country_codes = e.country_codes,
    incorporation_date = e.incorporation_date,
    sourceID = e.sourceID,
    inactivation_date = e.struck_off_date,
})
""",
    env=env,
    type_strict=False,
)

Rule5_A = Rule(
    """
MATCH (e:Entity)
WHERE NOT e:StruckOff
  AND e.service_provider IS NOT NULL
GENERATE
(ae = (e):ActiveEntity {
    name = e.name,
    country_codes = e.country_codes,
    incorporation_date = e.incorporation_date,
    sourceID = e.sourceID
})
-[():MANAGED_BY]->
(sp = (e.service_provider):ServiceProvider {
    name = e.service_provider
})
""",
    env=env,
    type_strict=False,
)

Rule5_B = Rule(
    """
MATCH (e:Entity:StruckOff)
WHERE e.service_provider IS NOT NULL
GENERATE
(ie = (e):InactiveEntity {
    name = e.name,
    country_codes = e.country_codes,
    incorporation_date = e.incorporation_date,
    sourceID = e.sourceID,
    inactivation_date = e.struck_off_date,
})
-[():MANAGED_BY]->
(sp = (e.service_provider):ServiceProvider {
    name = e.service_provider
})
""",
    env=env,
    type_strict=False,
)

In [6]:
from dtgraph.pg_schema.check_schema import check_schema

check_schema(
    [
        Rule1_A,
        Rule1_B,
        Rule2_A,
        Rule2_B,
        Rule3_A,
        Rule3_B,
        Rule4_A,
        Rule4_B,
        Rule5_A,
        Rule5_B
    ],
    target_schema,
)


--- Checking Rule ---
Source dictionary:
{'lhs': 'MATCH (e:Entity)\nWHERE NOT e:StruckOff', 'constructors': [{'alias': 'ae', 'ids': ['e'], 'labels': ['ActiveEntity'], 'properties': [{'key': 'name', 'value': 'e.name'}, {'key': 'country_codes', 'value': 'e.country_codes'}, {'key': 'incorporation_date', 'value': 'e.incorporation_date'}, {'key': 'sourceID', 'value': 'e.sourceID\n'}]}]}

--- Checking Rule ---
Source dictionary:
{'lhs': 'MATCH (e:Entity:StruckOff)', 'constructors': [{'alias': 'ie', 'ids': ['e'], 'labels': ['InactiveEntity'], 'properties': [{'key': 'name', 'value': 'e.name'}, {'key': 'country_codes', 'value': 'e.country_codes'}, {'key': 'incorporation_date', 'value': 'e.incorporation_date'}, {'key': 'sourceID', 'value': 'e.sourceID'}, {'key': 'inactivation_date', 'value': 'e.struck_off_date'}]}]}

--- Checking Rule ---
Source dictionary:
{'lhs': 'MATCH (e:Entity)-[r:registered_address]->(a:Address)\nWHERE NOT e:StruckOff AND a.address IS NOT NULL', 'constructors': [{'src': {

In [ ]:
my_transform = Transformation([Rule1_A, Rule1_B, Rule2_A, Rule2_B, Rule3_A, Rule3_B, Rule4_A, Rule4_B, Rule5_A, Rule5_B])
my_transform.apply_on(graph)

Index: Added 0 index, completed after 12 ms.
Rule: Added 0 labels, created 0 nodes, set 2576400 properties, created 0 relationships, completed after 13856 ms.


13856

### Abort Transformation

In [8]:
my_transform.abort()

Index: Removed 1 index, completed after 7 ms.
Abort: Deleted 772316 nodes, deleted 1154128 relationships, completed after 7096 ms.
